# Étape 02 — Collecte et préparation (YouTube)

Projet LB Prod. Ici je récupère les stats de la chaîne YouTube de l'artiste Wilson (abonnés, vues totales) et les stats de chacune de ses vidéos (vues, likes, commentaires), via l'API YouTube Data v3.

## Présentation du projet

**Nom** : Boni
**Prénom** : Loïc

**Quelle entreprise avez-vous imaginée ? Quel est son besoin ?**
Entreprise fictive : **LB Prod**, un label musical qui produit plusieurs artistes, dont Wilson. LB Prod a besoin de suivre la performance digitale de ses artistes sur YouTube (abonnés, vues, engagement par clip) pour décider où investir en promotion, repérer les titres qui fonctionnent le mieux et comprendre l'évolution de l'audience dans le temps.

**Quelle API pensez-vous utiliser ? Pourquoi celle-ci ?**
YouTube Data API v3. Choisie après avoir testé l'API Spotify, qui bloque désormais l'accès (même aux données publiques) si le compte créateur de l'app n'a pas un abonnement Premium actif. YouTube est gratuite, sans condition d'abonnement, et couvre bien le besoin (stats de chaîne + stats par vidéo, avec des dates permettant une analyse dans le temps).

**Quelles données allez-vous récupérer ? Et comment seront-elles utiles à votre entreprise ?**
Côté chaîne : nombre d'abonnés, vues totales, nombre de vidéos. Côté vidéos : titre, date de publication, vues, likes, commentaires, pour chacun des clips. Ça permet à LB Prod d'identifier les titres qui performent le mieux, de suivre la croissance de la chaîne dans le temps (par date de sortie), et d'orienter les décisions promo (quel type de contenu refaire, quand relancer une campagne).

## 1. Import des librairies

Il me faut : requests pour appeler l'API, pandas pour manipuler les données, dotenv pour charger les variables du fichier .env, et os pour lire ces variables une fois chargées (os.getenv). Ça évite d'écrire ma clé API en clair dans le code : si je publie ce notebook sur GitHub, seul le code est visible, pas la clé (elle reste dans .env, qui est exclu par .gitignore).

In [1]:
# on importe les librairies dont on a besoin
import pandas as pd
import requests
from dotenv import load_dotenv
import os

# on charge les variables du fichier .env (dont la clé API)
load_dotenv("../.env")
API_KEY = os.getenv("YOUTUBE_API_KEY")

## 2. Appel de l'API pour les infos de la chaîne

Endpoint à utiliser : /channels → me donne les infos et statistiques d'une chaîne à partir de son channel_id.

Channel ID de Wilson (trouvé via son handle @wilsonofficiel) : UCFXq9XWKAM4DMlPwQAiMcPg

Doc officielle : https://developers.google.com/youtube/v3/docs/channels

In [2]:
# identifiant de la chaîne de Wilson
CHANNEL_ID = "UCFXq9XWKAM4DMlPwQAiMcPg"

# on appelle l'API pour récupérer les infos de la chaîne
url = "https://www.googleapis.com/youtube/v3/channels"
params = {
    "part": "snippet,statistics",
    "id": CHANNEL_ID,
    "key": API_KEY,
}
response = requests.get(url, params=params)

## 3. Contrôle de la réponse

Même réflexe que pour CoinGecko : je vérifie le code HTTP (200 = OK) avant d'utiliser la réponse.

In [3]:
# on vérifie que l'appel a marché (200 = tout va bien)
print(response.status_code)
channel_json = response.json()
channel_json

200


{'kind': 'youtube#channelListResponse',
 'etag': '-9Wndg5EJI9cRLftgatv346VAP4',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '8xdy86HkBPiLO2zUSH-uQ2aeEQM',
   'id': 'UCFXq9XWKAM4DMlPwQAiMcPg',
   'snippet': {'title': 'Wilson',
    'description': 'Chaine Youtube officielle de Wilson\nOfficial Youtube channel of Wilson\n\nWilson de son nom complet Bourge Leadry Ntseba né le 15/08/1999 à Brasaville au Congo. \nEn 2011 il décide de poursuivre ses études en France.\nPassionné de musique depuis son plus jeune âge il est influencé par des artistes tel que : Koffi, Sexion d’assaut ou encore Fally Lpupa.\nEn 2018 il décide de lancer sa carrière musicale avec son premier single "Validé", qui cumule plus de 22 Millions de vues en 12 mois. Un début très prometteur, par la suite Wilson Sort "Kimono", "Brisé" "C’est compliqué » et “Piqué“ qui cumule déjà plus de 3M de vues\nEn peu de temps Wilson fait parler de lui dans le 

**Note pour moi** : status 200, bon signe. La réponse contient une liste "items" avec un seul élément (ma chaîne), avec un bloc "snippet" (titre, description...) et un bloc "statistics" (abonnés, vues, nb vidéos).

## 4. Transformation en DataFrame

Je ne garde que ce qui m'intéresse : titre de la chaîne, nombre d'abonnés, nombre de vues total, nombre de vidéos.

In [4]:
# on récupère seulement le premier (et seul) résultat
channel_info = channel_json["items"][0]

# on garde juste les infos utiles dans un tableau
channel_df = pd.DataFrame([{
    "channel_id": channel_info["id"],
    "title": channel_info["snippet"]["title"],
    "subscriber_count": channel_info["statistics"]["subscriberCount"],
    "view_count": channel_info["statistics"]["viewCount"],
    "video_count": channel_info["statistics"]["videoCount"],
}])
channel_df

,channel_id,title,subscriber_count,view_count,video_count
0,UCFXq9XWKAM4DMlPwQAiMcPg,Wilson,277000,99485456,68


## 5. Vérification des types

L'API renvoie les nombres sous forme de texte (str). Je dois les convertir en entiers pour pouvoir les utiliser dans des calculs plus tard.

In [5]:
# on regarde le type de chaque colonne
channel_df.dtypes

channel_id          str
title               str
subscriber_count    str
view_count          str
video_count         str
dtype: object

In [6]:
# on transforme le texte en vrais nombres entiers
channel_df["subscriber_count"] = channel_df["subscriber_count"].astype(int)
channel_df["view_count"] = channel_df["view_count"].astype(int)
channel_df["video_count"] = channel_df["video_count"].astype(int)

channel_df.dtypes

channel_id            str
title                 str
subscriber_count    int64
view_count          int64
video_count         int64
dtype: object

**Note pour moi** : les trois colonnes de stats étaient en str (comportement normal de l'API YouTube, elle renvoie tout en texte), maintenant en int64, prêtes pour des calculs.

## 6. Récupération de la liste des vidéos

L'API ne donne pas directement "toutes les vidéos d'une chaîne". Il faut d'abord récupérer l'ID de la playlist "uploads" de la chaîne (une playlist spéciale qui contient automatiquement toutes ses vidéos), puis lister son contenu.

In [7]:
# on demande à l'API l'identifiant de la playlist "uploads" de la chaîne
url = "https://www.googleapis.com/youtube/v3/channels"
params = {
    "part": "contentDetails",
    "id": CHANNEL_ID,
    "key": API_KEY,
}
response = requests.get(url, params=params)
print(response.status_code)

# cette playlist contient automatiquement toutes les vidéos de la chaîne
uploads_playlist_id = response.json()["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
uploads_playlist_id

200


'UUFXq9XWKAM4DMlPwQAiMcPg'

## 7. Liste des vidéos de la playlist uploads

L'API limite à 50 résultats par appel (maxResults). Si la chaîne a plus de 50 vidéos, il faut enchaîner les pages avec un nextPageToken, jusqu'à ce qu'il n'y en ait plus. C'est pour ça qu'il y a une boucle while ici.

In [8]:
video_ids = []
next_page_token = None

# l'API ne donne que 50 vidéos par appel, donc on boucle tant qu'il y a une page suivante
while True:
    url = "https://www.googleapis.com/youtube/v3/playlistItems"
    params = {
        "part": "contentDetails",
        "playlistId": uploads_playlist_id,
        "maxResults": 50,
        "pageToken": next_page_token,
        "key": API_KEY,
    }
    response = requests.get(url, params=params)
    page = response.json()

    # on ajoute les IDs trouvés sur cette page à notre liste
    for item in page["items"]:
        video_ids.append(item["contentDetails"]["videoId"])

    # s'il n'y a plus de page suivante, on arrête la boucle
    next_page_token = page.get("nextPageToken")
    if next_page_token is None:
        break

print(len(video_ids), "vidéos trouvées")

67 vidéos trouvées


**Note pour moi** : nombre de vidéos trouvées cohérent avec video_count de l'étape 4 (à 1 près parfois : une vidéo peut être privée/supprimée entre les deux appels).

## 8. Stats de chaque vidéo

Même logique que pour la chaîne, mais appliquée à chaque vidéo : vues, likes, commentaires, date de publication. L'endpoint /videos accepte jusqu'à 50 IDs à la fois, donc je demande les vidéos par lots de 50 (boucle for).

In [9]:
videos_data = []

# l'API accepte 50 IDs maximum par appel, donc on traite les vidéos par paquets de 50
for i in range(0, len(video_ids), 50):
    batch_ids = video_ids[i:i + 50]

    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part": "snippet,statistics",
        "id": ",".join(batch_ids),
        "key": API_KEY,
    }
    response = requests.get(url, params=params)
    print(response.status_code)

    # pour chaque vidéo, on garde les infos qui nous intéressent
    for video in response.json()["items"]:
        videos_data.append({
            "video_id": video["id"],
            "title": video["snippet"]["title"],
            "published_at": video["snippet"]["publishedAt"],
            "view_count": video["statistics"].get("viewCount", 0),
            "like_count": video["statistics"].get("likeCount", 0),
            "comment_count": video["statistics"].get("commentCount", 0),
        })

videos = pd.DataFrame(videos_data)
videos.head(10)

200


200


,video_id,title,published_at,view_count,like_count,comment_count
0,cWZPnt1H1J0,WILSON - BOSS LADY CLIP (OFFICIEL),2026-09-18T13:53:26Z,222542,547,21
1,BAZ6iyUOND0,Vous avez écoutez cette chanson ? 🍯🍯🎵 c’est du...,2026-03-01T16:22:44Z,3650,207,8
2,9nzHMgeR2yI,@WeisleyOfficiel on doit sortir cette chanson...,2026-02-14T20:10:26Z,3777,182,11
3,_IWy-D-N-_E,WILSON - PEUR D'AIMER (ZOUK REMIX)- DJ FAB,2026-02-05T23:09:31Z,67235,1334,40
4,Zaia7Lk4ASk,Est ce que vous voulez cette version du coup ?...,2026-01-29T17:44:21Z,4616,343,32
5,-Q29eYpeZlA,🍯😭🎵,2026-01-26T18:21:08Z,3285,273,26
6,500ZVpwyyQw,🍯🎵,2026-01-15T21:38:36Z,4240,235,12
7,Fld1IoltwQM,12 janvier 2026,2026-01-12T18:38:30Z,2815,144,8
8,5MaWgSZQY_w,À bon entendeur 🧏🏾‍♂️,2026-01-11T10:35:40Z,2508,170,9
9,lO3pAfk-WHU,WILSON - PEUR D'AIMER (CLIP OFFICIEL),2026-01-02T17:00:07Z,475669,7483,213


## 9. Vérification des types

In [10]:
# on vérifie le type de chaque colonne
videos.dtypes

video_id         str
title            str
published_at     str
view_count       str
like_count       str
comment_count    str
dtype: object

In [11]:
# texte -> nombres entiers
videos["view_count"] = videos["view_count"].astype(int)
videos["like_count"] = videos["like_count"].astype(int)
videos["comment_count"] = videos["comment_count"].astype(int)
# texte -> vraie date
videos["published_at"] = pd.to_datetime(videos["published_at"])

videos.dtypes

video_id                         str
title                            str
published_at     datetime64[us, UTC]
view_count                     int64
like_count                     int64
comment_count                  int64
dtype: object

**Note pour moi** : vues/likes/commentaires passés de str à int64, published_at passé de str à datetime. Mêmes corrections que pour last_updated dans le notebook CoinGecko.

## 10. Valeurs manquantes

In [12]:
# on compte les valeurs manquantes colonne par colonne
videos.isna().sum()

video_id         0
title            0
published_at     0
view_count       0
like_count       0
comment_count    0
dtype: int64

**Note pour moi** : aucune valeur manquante. Les commentaires désactivés sur une vidéo auraient pu donner un comment_count manquant, mais le .get(..., 0) utilisé à l'étape 8 évite le problème en mettant 0 par défaut.

## 11. Doublons

In [13]:
# on compte les lignes en double
videos.duplicated().sum()

np.int64(0)

**Note pour moi** : aucun doublon, chaque vidéo n'apparaît qu'une fois.

## 12. Horodatage de la collecte

J'ajoute une colonne collected_at, comme pour CoinGecko, pour suivre l'évolution dans le temps si je relance la collecte plus tard.

In [14]:
# on note l'heure de la collecte, utile pour suivre l'évolution dans le temps
channel_df["collected_at"] = pd.Timestamp.now(tz="UTC")
videos["collected_at"] = pd.Timestamp.now(tz="UTC")

## 13. Tri par popularité

Pour voir d'un coup d'œil les vidéos qui marchent le mieux.

In [15]:
# on trie les vidéos par nombre de vues, de la plus populaire à la moins populaire
videos = videos.sort_values("view_count", ascending=False).reset_index(drop=True)
videos.head(10)

,video_id,title,published_at,view_count,like_count,comment_count,collected_at
0,_rff6xFhb9E,WILSON - VALIDÉ (Clip Officiel),2018-10-19 16:30:01+00:00,37018775,212609,3063,2026-10-14 08:25:13.167446+00:00
1,A7aTBemow4c,WILSON - J'TE LE DIS (Clip Officiel),2020-05-31 15:02:02+00:00,13923659,124327,2932,2026-10-14 08:25:13.167446+00:00
2,DNDTI1xPl3M,WILSON - PIQUÉ (Clip Officiel),2019-06-13 16:33:44+00:00,7077311,85833,2372,2026-10-14 08:25:13.167446+00:00
3,ExqaybpnyKc,WILSON - KIMONO (Clip Officiel),2018-12-14 17:27:55+00:00,3608320,37408,691,2026-10-14 08:25:13.167446+00:00
4,JDwN0h1R7hY,WILSON - OUBLIE TON EX,2023-09-15 16:00:13+00:00,2630024,30430,866,2026-10-14 08:25:13.167446+00:00
5,Ce1o-bG8N90,WILSON - BRISÉ (Clip Officiel),2019-02-15 17:30:00+00:00,1895445,27399,522,2026-10-14 08:25:13.167446+00:00
6,2gBL-xi_1xw,WILSON - On s'en ira (Clip Officiel),2022-04-24 16:00:14+00:00,1567033,21442,694,2026-10-14 08:25:13.167446+00:00
7,ri3dr_VJdlc,WILSON - J'AI DÉJA (Clip Officiel),2019-09-20 16:00:19+00:00,1516575,24182,659,2026-10-14 08:25:13.167446+00:00
8,VLeI1BM0j9c,WILSON - KARABA (Clip Officiel),2021-05-06 16:00:15+00:00,1477838,18225,447,2026-10-14 08:25:13.167446+00:00
9,qql1QNMOHWE,WILSON - RAPPELLE-TOI (Clip Officiel),2021-02-12 17:00:15+00:00,1342778,17062,404,2026-10-14 08:25:13.167446+00:00


## 14. Export du résultat

In [16]:
# on sauvegarde les deux tableaux en fichiers CSV
channel_df.to_csv("../data/channel_profile.csv", index=False)
videos.to_csv("../data/videos.csv", index=False)
print("Fichiers exportés dans data/")

Fichiers exportés dans data/


## Bilan

**Note pour moi** : appel API YouTube OK (status 200). Chaîne Wilson : 277 000 abonnés, ~99,5M vues au total, 68 vidéos. Toutes les vidéos récupérées (playlist uploads + pagination), stats converties en bons types, aucune valeur manquante ni doublon, collected_at ajoutée. Vidéo la plus vue : "VALIDÉ" (37M vues). Export fait dans data/channel_profile.csv et data/videos.csv.